In [ ]:
import pandas as pd

# read data
oct_2019 = pd.read_csv("/Users/katychuang/Documents/Projects/ecommerce_behavior_data/2019-Oct.csv")

print(oct_2019.shape)
print(oct_2019.info())
print(oct_2019.head())

In [14]:
# transform event date to datetime and extract the date only
# oct_2019['event_time'] = pd.to_datetime(oct_2019['event_time']).dt.date

# check if there's any unreasonable price (negative price)
(oct_2019["price"] < 0).any()

# remove duplicates
oct_2019 = oct_2019.drop_duplicates()

In [ ]:
# Since currernt data is event-level, we need to transform the data into various data level based on the analysis we want to do.

# session-level analysis table
oct_2019['purchase_revenue'] = oct_2019['price'] * (oct_2019['event_type'] == 'purchase')  

session_df = oct_2019.groupby(['user_id', 'user_session']).agg(
    session_start = ('event_time', 'min'),
    session_end = ('event_time', 'max'),
    total_events = ('event_type', 'count'),
    view_count = ('event_type', lambda x: (x == 'view').sum()),
    cart_count = ('event_type', lambda x: (x == 'cart').sum()),
    purchase_count = ('event_type', lambda x: (x == 'purchase').sum()),
    total_revenue = ('purchase_revenue', 'sum')
).reset_index()

# session action
session_df['has_view'] = (session_df['view_count'] > 0).astype(int)
session_df['has_cart'] = (session_df['cart_count'] > 0).astype(int)
session_df['has_purchase'] = (session_df['purchase_count'] > 0).astype(int)

# session duration
#session_df['session_duration_sec'] = (
#    session_df['session_end'] - session_df['session_start']
#).dt.total_seconds()

session_df[session_df['has_purchase'] > 0].head()

In [ ]:
# user-level analysis table
user_df = (session_df.groupby('user_id').agg(
    total_sessions = ('user_session', 'nunique'),
    total_views = ('view_count', 'sum'),
    total_carts = ('cart_count', 'sum'),
    total_purchases = ('purchase_count', 'sum'),
    total_revenue = ('total_revenue', 'sum')
)).reset_index()

user_df[user_df['total_sessions'] > 1].head()

# adding if user has purchase with 1 = true, 0 = false
user_df['has_purchase'] = (user_df['total_purchases'] > 0).astype(int)

In [ ]:
# funnel table
funnel = pd.DataFrame(
    {
        'stage' : ['view', 'cart', 'purchase'],
        'sessions' : [
            session_df['has_view'].sum(),  # counts of sessions which has view
            session_df['has_cart'].sum(),  # counts of sessions which has cart
            session_df['has_purchase'].sum()  # counts of purchase which has purchase
        ]
    }
)

funnel.head()


# conversion rate
view_sessions = funnel.loc[funnel['stage'] == 'view', 'sessions'].iloc[0]
funnel['conversion_rate'] = funnel['sessions'] / view_sessions

print(funnel)